# 04. Exportar associações

Mesma regra do [`03_avaliar.ipynb`](03_avaliar.ipynb): melhor CPF por Censo
entre pares com `p ≥ THRESHOLD_AVALIACAO`; empate por CEP + `nome_mae_phon`;
grava CPFs com **até** `MAX_CENSOS_POR_CPF` Censos (4+ anula o grupo). Sem
cluster, sem greedy, sem métricas de ouro — avaliação fica no 03.

Saída: `splink_atribuicao.parquet`.


In [ ]:
import sys
from pathlib import Path

PROB_DIR = Path.cwd()
if PROB_DIR.name == 'notebooks':
    PROB_DIR = PROB_DIR.parent
if str(PROB_DIR) not in sys.path:
    sys.path.insert(0, str(PROB_DIR))

from config import (
    MAX_CENSOS_POR_CPF,
    SPLINK_ATRIBUICAO,
    SPLINK_INPUT_VIEW,
    SPLINK_PREDICTIONS,
    TABELA_CENSO_LIMPA_APLICACAO,
    TABELA_CPF_LIMPA_APLICACAO,
    THRESHOLD_AVALIACAO,
    drop_splink_temp_tables,
    export_parquet,
    get_connection,
    materialize_splink_input,
    print_paths,
    require_input,
    require_tables,
)

T = THRESHOLD_AVALIACAO

print_paths()
require_input(SPLINK_PREDICTIONS, label='SPLINK_PREDICTIONS (rode o 02b_aplicar antes)')
con = get_connection()
drop_splink_temp_tables(con)
require_tables(
    con,
    [TABELA_CENSO_LIMPA_APLICACAO, TABELA_CPF_LIMPA_APLICACAO],
    notebook_origem='00b',
)
materialize_splink_input(
    con,
    censo_table=TABELA_CENSO_LIMPA_APLICACAO,
    cpf_table=TABELA_CPF_LIMPA_APLICACAO,
)
print('T:', T, '| max Censos/CPF:', MAX_CENSOS_POR_CPF)


In [ ]:
_tipo = con.execute('''
SELECT table_type
FROM information_schema.tables
WHERE table_schema = 'main' AND table_name = 'splink_predictions'
''').fetchone()
if _tipo:
    _kind = 'VIEW' if _tipo[0].upper() == 'VIEW' else 'TABLE'
    con.execute(f'DROP {_kind} IF EXISTS splink_predictions')
con.execute(f'''
CREATE OR REPLACE VIEW splink_predictions AS
SELECT
    CASE
        WHEN unique_id_l LIKE 'censo_%' THEN unique_id_l
        ELSE unique_id_r
    END AS unique_id_censo,
    CASE
        WHEN unique_id_l LIKE 'censo_%' THEN unique_id_r
        ELSE unique_id_l
    END AS unique_id_cpf,
    match_probability
FROM read_parquet('{SPLINK_PREDICTIONS}')
''')

con.execute(f'''
CREATE OR REPLACE TABLE melhor_por_censo AS
SELECT p.unique_id_censo, p.unique_id_cpf, p.match_probability
FROM splink_predictions p
JOIN {SPLINK_INPUT_VIEW} ca ON ca.unique_id = p.unique_id_censo
JOIN {SPLINK_INPUT_VIEW} pb ON pb.unique_id = p.unique_id_cpf
WHERE p.match_probability >= {T}
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY p.unique_id_censo
    ORDER BY
        p.match_probability DESC,
        (
            CAST(ca.cep IS NOT NULL AND pb.cep IS NOT NULL AND ca.cep = pb.cep AS INTEGER)
            + CAST(
                ca.nome_mae_phon IS NOT NULL AND pb.nome_mae_phon IS NOT NULL
                AND ca.nome_mae_phon = pb.nome_mae_phon AS INTEGER
            )
        ) DESC,
        p.unique_id_cpf
) = 1
''')

con.execute(f'''
CREATE OR REPLACE TABLE associacoes_unicas AS
SELECT m.*
FROM melhor_por_censo m
JOIN (
    SELECT unique_id_cpf
    FROM melhor_por_censo
    GROUP BY 1
    HAVING COUNT(*) <= {MAX_CENSOS_POR_CPF}
) c ON c.unique_id_cpf = m.unique_id_cpf
''')

n_melhor = con.execute('SELECT COUNT(*) FROM melhor_por_censo').fetchone()[0]
n_unicas = con.execute('SELECT COUNT(*) FROM associacoes_unicas').fetchone()[0]
print('Censos com par >= T:', f'{n_melhor:,}')
print('Associações na lista (n_censo <=', MAX_CENSOS_POR_CPF, '):', f'{n_unicas:,}')


In [ ]:
export_parquet(con, 'associacoes_unicas', path=SPLINK_ATRIBUICAO)
print('Exportado:', SPLINK_ATRIBUICAO)
con.close()
